# FNO for PDEs — Part 2: Train on 2D Navier-Stokes (Colab)

Trains FNO2d and a U-Net2d baseline on the vorticity dataset and compares them
(the 2D analogue of Part 1's accuracy experiment). Runs on the GPU.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**.


## 1. Clone and install

In [ ]:
!git clone https://github.com/amark-23/fno-pde.git
%cd fno-pde
!pip install -q -r requirements.txt
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")


## 2. Get the dataset

Loads `navier_stokes.npz` from Google Drive if you saved it there; otherwise
regenerates it on the GPU (about a minute).


In [ ]:
import os, numpy as np
DATA = "data/navier_stokes.npz"; os.makedirs("data", exist_ok=True)
try:
    from google.colab import drive; drive.mount('/content/drive')
    src = "/content/drive/MyDrive/navier_stokes.npz"
    if os.path.exists(src):
        import shutil; shutil.copy(src, DATA); print("loaded from Drive")
except Exception as e:
    print("Drive skip:", e)
if not os.path.exists(DATA):
    from src.solvers.navier_stokes_torch import generate_dataset
    print("generating..."); d = generate_dataset(n_traj=1000, N=64, nu=1e-3, dt=1e-3, t_final=15.0, n_snapshots=20, seed=0)
    np.savez_compressed(DATA, vorticity=d); print("generated", d.shape)
print("dataset:", np.load(DATA)["vorticity"].shape)


## 3. Load train/test

In [ ]:
from src.data_ns import load_ns, make_loader
(x_tr, y_tr), (x_te, y_te), (x_norm, y_norm) = load_ns(DATA, n_train_traj=800, n_test_traj=200)
print("train pairs:", x_tr.shape[0], "| test pairs:", x_te.shape[0], "| input", tuple(x_tr.shape[1:]))


## 4. Training helpers

In [ ]:
import torch, time
from src.metrics import relative_l2
device = "cuda" if torch.cuda.is_available() else "cpu"

def predict(model, x, bs=64):
    model.eval(); outs=[]
    with torch.no_grad():
        for i in range(0, len(x), bs):
            outs.append(model(x[i:i+bs].to(device)).cpu())
    return torch.cat(outs)

def train(model, epochs=50, lr=1e-3, bs=32):
    model = model.to(device)
    loader = make_loader(x_tr, y_tr, batch_size=bs)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=max(1,epochs//2), gamma=0.5)
    t0=time.time()
    for ep in range(epochs):
        model.train(); tot=nb=0
        for xb,yb in loader:
            xb,yb=xb.to(device),yb.to(device)
            opt.zero_grad(); loss=relative_l2(model(xb),yb); loss.backward(); opt.step()
            tot+=loss.item(); nb+=1
        sched.step()
        if (ep+1)%max(1,epochs//10)==0: print(f"  epoch {ep+1}/{epochs}  train rel-L2 {tot/nb:.4f}")
    pred = y_norm.decode(predict(model, x_te))
    err = relative_l2(pred, y_te).item()
    n = sum(p.numel() for p in model.parameters())
    print(f"  test rel-L2 {err:.4f} | params {n:,} | {time.time()-t0:.0f}s")
    return model, err


## 5. Train FNO2d and U-Net2d

In [ ]:
from src.models.fno import FNO2d
from src.models.unet import UNet2d

print("FNO2d:")
fno, fno_err = train(FNO2d(modes1=12, modes2=12, width=32, depth=4, in_channels=3, out_channels=1))
print("U-Net2d:")
unet, unet_err = train(UNet2d(in_channels=3, out_channels=1, width=24, n_levels=3))

print(f"\nAccuracy (single-step, lower=better):  FNO {fno_err:.4f}   U-Net {unet_err:.4f}")


## 6. Visualize a prediction

In [ ]:
import matplotlib.pyplot as plt
i = 0
inp = x_te[i:i+1]
truth = y_te[i,:,:,0].numpy()
pred = y_norm.decode(predict(fno, inp))[0,:,:,0].numpy()
fig, ax = plt.subplots(1, 3, figsize=(13,4))
for a,(img,t) in zip(ax, [(x_norm.decode(inp[0,:,:,0]).numpy(),"input w(t)"),(truth,"truth w(t+1)"),(pred,"FNO w(t+1)")]):
    im=a.imshow(img, cmap="RdBu_r", origin="lower"); a.set_title(t); a.set_xticks([]); a.set_yticks([]); fig.colorbar(im,ax=a,fraction=0.046)
plt.tight_layout(); plt.show()


## Next steps
- Resolution transfer (train at 64, evaluate at 32 via spectral downsampling).
- Autoregressive rollout: apply the model repeatedly, measure error growth.
- Speed: FNO2d inference vs the numerical solver.
